In [2]:
%pip install openai

  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
   ---------------------------------------- 0.0/683.3 kB ? eta -:--:--
   - -------------------------------------- 30.7/683.3 kB 1.3 MB/s eta 0:00:01
   -- ------------------------------------ 41.0/683.3 kB 653.6 kB/s eta 0:00:01
   ---- ---------------------------------- 71.7/683.3 kB 653.6 kB/s eta 0:00:01
   ------ ------------------------------- 112.6/683.3 kB 656.4 kB/s eta 0:00:01
   ------ ------------------------------- 112.6/683.3 kB 656.4 kB/s eta 0:00:01
   ------- ------------------------------ 143.4/683.3 kB 532.5 kB/s eta 0:00:02
   --------- ---------------------------- 174.1/683.3 kB 551.6 kB/s eta 0:00:01
   ---------- --------------------------- 194.6/683.3 kB 562.0 kB/s eta 0:00:01
   ----------- -------------------------- 204.8/683.3 kB 518.8 kB/s eta 0:00:01
   ------------- ------------------------ 235.5/683.3 kB 533.8 kB/s eta 0:00:01
   -------------- ----------------------- 256.0/683.3 kB 524.0 kB/s


[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Import Necessary Libraries

In [3]:
import re
from collections import Counter
import nltk
from nltk.corpus import stopwords
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from openai import OpenAI

download nltk library

In [4]:
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\arafa\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Openai api key

In [ ]:
client = OpenAI(api_key="your_openai_api_key")

Open file with File read

In [6]:
with open("ChatLog.txt", "r") as f:
    chatlog = f.read()

In [7]:
words = re.findall(r'\b\w+\b', chatlog.lower())
filtered_words = [word for word in words if word not in stop_words and not word.isdigit()]
word_freq = Counter(filtered_words)
top_5_keywords = [word for word, freq in word_freq.most_common(5)]

Splitting

In [8]:
exchanges = [exchange.strip() for exchange in chatlog.split('\n\n') if exchange.strip()]
total_exchanges = len(exchanges)

Embedding

In [9]:
exchange_embeddings = []
for exchange in exchanges:
    response = client.embeddings.create(model="text-embedding-ada-002", input=exchange)
    embedding = response.data[0].embedding
    exchange_embeddings.append(embedding)
exchange_embeddings = np.array(exchange_embeddings)

In [10]:
query = "What is the nature of the conversation?"
response = client.embeddings.create(model="text-embedding-ada-002", input=query)
query_embedding = np.array(response.data[0].embedding)

Check Similarities

In [11]:
similarities = cosine_similarity([query_embedding], exchange_embeddings)[0]
top_k_indices = np.argsort(similarities)[-5:][::-1]  # Top 5
top_exchanges = [exchanges[i] for i in top_k_indices]
context = "\n\n".join(top_exchanges)

Prompt Engineering

In [16]:
prompt = f"Based on the following information, describe the nature of the conversation:\n\n{context}"
response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[{"role": "user", "content": prompt}]
)
nature_of_conversation = response.choices[0].message.content

Summary

In [17]:
summary = (
    f"Summary:\n"
    f"- The conversation had {total_exchanges} exchanges.\n"
    f"- {nature_of_conversation}\n"
    f"- Most common keywords: {', '.join(top_5_keywords)}"
)
print(summary)

Summary:
- The conversation had 200 exchanges.
- The nature of the conversation is repetitive and similar for each topic mentioned. The AI provides a brief overview of the topic and mentions that it can be used in various areas such as education, research, and practical applications. The conversation does not delve into specific details about the topics but highlights their versatility and usefulness across different domains.
- Most common keywords: topic, user, ai, use, hi
